In [1]:
!pip install -q unsloth gradio huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive"

!cp "{DRIVE_ROOT}/llama3.1-8b-legal-clause-lora-dpo.zip" .
!unzip -q llama3.1-8b-legal-clause-lora-dpo.zip
!ls llama3.1-8b-legal-clause-lora-dpo

!mkdir -p data
!cp "{DRIVE_ROOT}/categories.json" data/ 2>/dev/null
!cp "{DRIVE_ROOT}/data/categories.json" data/ 2>/dev/null
!ls data/

Mounted at /content/drive
adapter_config.json	   chat_template.jinja	tokenizer_config.json
adapter_model.safetensors  README.md		tokenizer.json
categories.json


In [3]:
import json

with open("data/categories.json") as f:
    cat_config = json.load(f)

CATEGORIES = cat_config["categories"]
SYSTEM_PROMPT = cat_config["system_prompt"]
print("Categories:", CATEGORIES)

Categories: ['Governing Laws', 'Terminations', 'Confidentiality', 'Indemnifications', 'Assignments', 'Notices', 'Severability', 'Counterparts', 'Amendments', 'Survival']


In [4]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

MAX_SEQ_LENGTH = 1024
DPO_ADAPTER_PATH = "llama3.1-8b-legal-clause-lora-dpo"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=DPO_ADAPTER_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

MERGED_DIR = "llama3.1-8b-legal-clause-merged-16bit"
model.save_pretrained_merged(
    MERGED_DIR, tokenizer, save_method="merged_16bit",
    maximum_memory_usage=0.5,   # caps GPU memory used during the merge step, to avoid
                                  # OOM on a T4 (lower = safer but slightly slower merge)
)

print(f"Merged model saved to: {MERGED_DIR}")
!du -sh {MERGED_DIR}

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load llama3.1-8b-legal-clause-lora-dpo as a legacy tokenizer.
Unsloth 2026.7.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [03:29<00:00, 52.47s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [03:47<00:00, 56.77s/it]


Unsloth: Merge process complete. Saved to `/content/llama3.1-8b-legal-clause-merged-16bit`
Merged model saved to: llama3.1-8b-legal-clause-merged-16bit
15G	llama3.1-8b-legal-clause-merged-16bit


In [5]:
from huggingface_hub import login
login()   # paste your HF token when prompted (needs "write" access)

HF_USERNAME = "kaustubh67"   # <-- change this to your actual HF username
HF_REPO_NAME = "llama3.1-8b-legal-clause-classifier"
HF_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"

model.push_to_hub_merged(
    HF_REPO_ID,
    tokenizer,
    save_method="merged_16bit",
)

print(f"Model pushed to: https://huggingface.co/{HF_REPO_ID}")

Unsloth: Restored added_tokens_decoder metadata in kaustubh67/llama3.1-8b-legal-clause-classifier/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0004-of-00004.safetensors:   3%|2         | 32.0MB / 1.17GB            


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [06:07<00:00, 91.85s/it]


Unsloth: Merge process complete. Saved to `/content/kaustubh67/llama3.1-8b-legal-clause-classifier`
Model pushed to: https://huggingface.co/kaustubh67/llama3.1-8b-legal-clause-classifier


In [6]:
BASE_MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
base_tokenizer = get_chat_template(base_tokenizer, chat_template="llama-3.1")
FastLanguageModel.for_inference(base_model)
FastLanguageModel.for_inference(model)   # our merged fine-tuned model, from Cell 4

print("Both base and fine-tuned models loaded and ready.")

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.


Both base and fine-tuned models loaded and ready.


In [7]:
import torch

def classify(clause_text, use_model, use_tokenizer):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Clause:\n{clause_text}"},
    ]
    inputs = use_tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        outputs = use_model.generate(
            input_ids=inputs, max_new_tokens=12, temperature=0.0, do_sample=False,
            pad_token_id=use_tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs.shape[-1]:]
    raw = use_tokenizer.decode(generated, skip_special_tokens=True).strip()
    for cat in CATEGORIES:
        if cat.lower() in raw.lower():
            return cat
    return f"(unrecognized output: {raw})"

def compare_models(clause_text):
    base_pred = classify(clause_text, base_model, base_tokenizer)
    finetuned_pred = classify(clause_text, model, tokenizer)
    return base_pred, finetuned_pred

# quick sanity check
sample = "This Agreement shall be governed by the laws of the State of New York."
print("Base:", classify(sample, base_model, base_tokenizer))
print("Fine-tuned:", classify(sample, model, tokenizer))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=12) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

Base: Governing Laws


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Fine-tuned: Governing Laws


In [15]:
import gradio as gr

EXAMPLE_CLAUSES = [
    # Verified from Step 04 evaluation: base model gets this wrong (distracted by the
    # word "terminated"), fine-tuned model gets it right.
    "This Agreement shall not be amended, modified or supplemented at any time or "
    "terminated without the prior written consent of all parties.",
    # Verified from Step 04: base model defaults to "Governing Laws" here, fine-tuned
    # correctly identifies it as Severability.
    "Any provision of this Agreement that is prohibited or unenforceable in any "
    "jurisdiction will be ineffective only as to that jurisdiction.",
    # A clean, easy example for contrast — both models likely agree here.
    "This Agreement shall be governed by the laws of the State of Delaware.",
]

def gradio_classify(clause_text):
    if not clause_text.strip():
        return "—", "—"
    base_pred, finetuned_pred = compare_models(clause_text)
    return base_pred, finetuned_pred

demo = gr.Interface(
    fn=gradio_classify,
    inputs=gr.Textbox(lines=6, label="Paste a contract clause",
                       placeholder="e.g. This Agreement shall be governed by the laws of..."),
    outputs=[
        gr.Textbox(label="Base Llama 3.1 8B (no fine-tuning)"),
        gr.Textbox(label="Fine-tuned (SFT + DPO)"),
    ],
    examples=EXAMPLE_CLAUSES,
    title="Legal Clause Classifier — Base vs Fine-tuned",
    description=(
        "Compares a plain Llama 3.1 8B model against our LoRA+QLoRA fine-tuned version "
        "(SFT + DPO) on legal clause classification. Categories: "
        + ", ".join(CATEGORIES)
    ),
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://110139d18aa5254e54.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
